# 🔬 NÂNG CẤP CHUẨN HÓA `BalancedPrototypeHead-256` CHO 4 MÔ HÌNH PROTOTYPE (`M4, M6, M7, M8` — `16k × bs=8`)

> **Mục tiêu:** Thay thế `LitePrototypeHead (128ch DW-Sep)` bị cắt quá tay bằng **`BalancedPrototypeHead-256`** để đạt **CẢ HAI mục tiêu: giữ/tăng IoU ngang hoặc vượt `M1/M3` ĐỒNG THỜI giảm 3.1× tổng FLOPs (`231.5 → ~73 GFLOPs`) & nhanh hơn ~2.1× CPU (`~3.6s` Full-Scan 8 tiles)**:
> * **Kiến trúc Decoder chuẩn `MiT-B0`:** Giữ trọn **`channels=256`, `dim=128`, 2 lớp FULL Dense `3×3 (256→256)`** ở độ phân giải gốc `1/4` (`128×128`), cắt hoàn toàn `Conv 3×3` dư thừa ở `256×256` & `cc_proj` nặng (`Head: 196 → 49.5 GFLOPs`, giảm 4×!).
> * **Sửa 2 lỗi toán học trong Prototype Loss:**
>   1. **Class-Balanced Prototype InfoNCE** (`0.5·CE_auth + 0.5·CE_tamp`): không để 99% nền đè bẹp 1% pixel giả mạo.
>   2. **Real-Gradient Inter-Class Separation Loss:** truyền gradient thật ép tâm đặc trưng 2 lớp tách nhau ra (không còn bị dính `cos ≈ 0.977` như trước!).
>   3. **Learnable `proto_fuse_weight` (`init=0.25`)**: mạng tự điều chỉnh trọng số trộn Prototype tối ưu.
>
> **Tiết kiệm GPU tối đa:** Giữ nguyên 100% kết quả của **`M1, M2, M3, M5`** đã chạy — chỉ chạy lại đúng **4 bản có Prototype (`M4, M6, M7, M8`) song song 1 lượt trên G4** (~33 GB / 96 GB VRAM, ~65–75 phút), đến Bước 3 **tự ghép thành bảng Full-Factorial $2^3 = 8$ dòng hoàn chỉnh**!

### 🟢 BƯỚC 1: KẾT NỐI DRIVE, CÀI MÔI TRƯỜNG & GIẢI NÉN DATASET (~3 Phút)

In [ ]:
import os, sys, glob, torch
from google.colab import drive

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"🖥️ GPU: {props.name} | VRAM: {props.total_memory/(1024**3):.1f} GB | BF16: {torch.cuda.is_bf16_supported()}")
    print(f"   CPU cores: {os.cpu_count()} | PyTorch: {torch.__version__}")
else:
    raise RuntimeError("❌ Chưa bật GPU Runtime! Hãy chọn Runtime -> Change runtime type -> G4 GPU.")

drive.mount("/content/drive")

%cd /content
!rm -rf TMD
!git clone https://github.com/gagaduy/TMD.git

sys.path.insert(0, "/content/TMD/ASCFormer")
%env PYTHONPATH=/content/TMD/ASCFormer

!pip install -q yapf==0.40.1 mmcv-lite==2.0.0 mmengine==0.7.0 timm einops prettytable rich onnx onnxruntime gdown

GDRIVE_ZIP_URL = "https://drive.google.com/file/d/1uBtDzuTcqAFX9vWUH3n8Vn3EewWpW5kP/view?usp=drivesdk"

os.makedirs("/content/TMD/ASCFormer/data/ttd", exist_ok=True)
drive_zip = "/content/drive/MyDrive/RealTextMan.zip"
if not os.path.exists(drive_zip):
    matches = glob.glob("/content/drive/MyDrive/**/RealTextMan.zip", recursive=True)
    if matches:
        drive_zip = matches[0]
    elif GDRIVE_ZIP_URL.strip():
        drive_zip = "/content/RealTextMan.zip"
        print("🌐 Không thấy trên MyDrive, đang tải RealTextMan.zip từ link chia sẻ bằng gdown...")
        !gdown --fuzzy "{GDRIVE_ZIP_URL.strip()}" -O "{drive_zip}"
    else:
        raise FileNotFoundError("❌ Không thấy RealTextMan.zip trên MyDrive và chưa có GDRIVE_ZIP_URL!")

print(f"📦 Đang giải nén {drive_zip} sang SSD Colab...")
!unzip -q -o "{drive_zip}" -d /content/TMD/ASCFormer/data/ttd/

ocr_cnt = len(glob.glob("/content/TMD/ASCFormer/data/ttd/RealTextMan/ocr_spatial/*.png"))
img_cnt = len(glob.glob("/content/TMD/ASCFormer/data/ttd/RealTextMan/JPEGImages/*.jpg"))
print(f"✅ Dataset sẵn sàng: {img_cnt} ảnh JPEGImages, {ocr_cnt} bản đồ ocr_spatial thật!")
assert img_cnt > 0 and ocr_cnt > 0

### 🔥 BƯỚC 2: HUẤN LUYỆN SONG SONG 4 BẢN `BalancedPrototypeHead-256` (`M4, M6, M7, M8`, `16k × bs=8`)
> * Giữ nguyên 100% `M1, M2, M3, M5` trên Drive — lưu 4 thư mục con mới `*_BalancedProto_256` trong `/content/drive/MyDrive/RTM_Ablation_B0_16k_bs8_G4/`.
> * Chạy song song **cả 4 model 1 lượt** (`MAX_PARALLEL_TRAIN = 4`, ~34 GB / 96 GB VRAM), tự backup checkpoint trung gian `4k, 8k, 12k, best_mIoU` và tự `--resume` nếu ngắt phiên.

In [ ]:
import os, sys, glob, time, re, shutil, subprocess, torch

%cd /content/TMD
!git pull origin main
%cd /content/TMD/ASCFormer
%env PYTHONPATH=/content/TMD/ASCFormer

_ = torch.hub.load_state_dict_from_url(
    "https://download.openmmlab.com/mmsegmentation/v0.5/pretrain/segformer/mit_b0_20220624-7e0fe6dd.pth",
    map_location="cpu"
)

DRIVE_ROOT = "/content/drive/MyDrive/RTM_Ablation_B0_16k_bs8_G4"
LOCAL_ROOT = "/content/work_dirs/ablation_b0_16k_bs8_balproto"
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(LOCAL_ROOT, exist_ok=True)

vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
MAX_PARALLEL_TRAIN = 4 if vram_gb >= 60 else (2 if vram_gb >= 30 else 1)
print(f"⚡ G4 Scheduler (BalancedProto M4/M6/M7/M8, 16k x bs=8): VRAM={vram_gb:.1f} GB -> MAX_PARALLEL_TRAIN = {MAX_PARALLEL_TRAIN}")

ABLATION_SUITE = [
    ("M4_BalancedProto_256",                  "configs/ascformer/ablation_b0_16k_bs8_m4_balproto.py"),
    ("M6_RealOCR_BalancedProto_256",          "configs/ascformer/ablation_b0_16k_bs8_m6_ocr_balproto.py"),
    ("M7_Focused_Rotate90_BalancedProto_256", "configs/ascformer/ablation_b0_16k_bs8_m7_focused_balproto.py"),
    ("M8_Full_OCR_Focused_BalancedProto_256", "configs/ascformer/ablation_b0_16k_bs8_m8_full_balproto.py"),
]

def backup_to_drive(local_dir, drive_dir, synced_set=None):
    os.makedirs(drive_dir, exist_ok=True)
    for ext in ("*.pth", "*.log", "*.json", "*.py", "*.png"):
        for fp in glob.glob(os.path.join(local_dir, ext)):
            bn = os.path.basename(fp)
            if synced_set is not None and bn.endswith(".pth"):
                sig = (bn, os.path.getsize(fp), int(os.path.getmtime(fp)))
                if sig in synced_set:
                    continue
                shutil.copy2(fp, os.path.join(drive_dir, bn))
                synced_set.add(sig)
            else:
                shutil.copy2(fp, os.path.join(drive_dir, bn))
    for fp in glob.glob(os.path.join(local_dir, "*", "*.log")) + glob.glob(os.path.join(local_dir, "*", "vis_data", "*.json")):
        shutil.copy2(fp, os.path.join(drive_dir, os.path.basename(fp)))
    for fp in glob.glob(os.path.join(local_dir, "**", "confusion_matrix.png"), recursive=True):
        shutil.copy2(fp, os.path.join(drive_dir, "confusion_matrix.png"))

def parse_progress(log_path):
    if not os.path.exists(log_path):
        return "Đang khởi tạo..."
    try:
        with open(log_path, "r", errors="ignore") as f:
            lines = f.readlines()[-40:]
        for ln in reversed(lines):
            if "Iter(train)" in ln:
                m = re.search(r"Iter\(train\)\s*\[\s*(\d+/\d+)\].*?eta:\s*([0-9:]+).*?loss:\s*([0-9.]+)", ln)
                if m:
                    return f"Iter [{m.group(1)}] | loss={m.group(3)} | ETA={m.group(2)}"
            if "Iter(val)" in ln:
                return "Đang chạy Valid 1,203 ảnh..."
    except Exception:
        pass
    return "Đang nạp dữ liệu / compile..."

queue = []
for name, cfg_path in ABLATION_SUITE:
    d_dir = os.path.join(DRIVE_ROOT, name)
    if os.path.exists(os.path.join(d_dir, "iter_16000.pth")):
        print(f"✅ [{name}] Đã hoàn tất 16,000 iters trên Drive — bỏ qua!")
    else:
        queue.append((name, cfg_path))

running = {}
synced_map = {}
failed = []
t_global = time.time()

WORKER_PY = "/tmp/run_train_worker_balproto.py"
with open(WORKER_PY, "w") as wf:
    wf.write("""import os, sys, runpy, torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
_orig_load = torch.load
def _safe_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _orig_load(*args, **kwargs)
torch.load = _safe_load
try:
    import yapf.yapflib.yapf_api as _y
    import mmengine.config.config as _m
    _orig_fc = _y.FormatCode
    def _safe_fc(*args, **kwargs):
        kwargs.pop('verify', None)
        return _orig_fc(*args, **kwargs)
    _y.FormatCode = _safe_fc
    _m.FormatCode = _safe_fc
except Exception:
    pass
sys.argv = ['tools/train.py'] + sys.argv[1:]
runpy.run_path('tools/train.py', run_name='__main__')
""")

while queue or running:
    while queue and len(running) < MAX_PARALLEL_TRAIN:
        name, cfg_path = queue.pop(0)
        l_dir = os.path.join(LOCAL_ROOT, name)
        d_dir = os.path.join(DRIVE_ROOT, name)
        os.makedirs(l_dir, exist_ok=True)
        os.makedirs(d_dir, exist_ok=True)

        d_ckpts = sorted(glob.glob(os.path.join(d_dir, "iter_*.pth")), key=os.path.getmtime)
        resume_args = []
        if d_ckpts:
            dst = os.path.join(l_dir, os.path.basename(d_ckpts[-1]))
            shutil.copy2(d_ckpts[-1], dst)
            resume_args = ["--resume", dst]

        log_path = os.path.join(l_dir, f"{name}_stdout.log")
        fh = open(log_path, "w")

        cmd = [sys.executable, WORKER_PY, cfg_path, "--work-dir", l_dir] + resume_args
        env = os.environ.copy()
        env["PYTHONPATH"] = "/content/TMD/ASCFormer"
        env["OMP_NUM_THREADS"] = "2"
        proc = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, env=env)
        running[name] = (proc, fh, log_path, l_dir, d_dir, time.time())
        synced_map[name] = set()
        print(f"🚀 Khởi chạy [{name}] (PID={proc.pid}) | Resume={resume_args}")

    time.sleep(30)
    elapsed = (time.time() - t_global) / 60.0
    gpu_stat = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu", "--format=csv,noheader"],
        text=True
    ).strip()
    print(f"\n⏱️ [{elapsed:.1f}p] G4 GPU (VRAM / Util): {gpu_stat}")

    finished_names = []
    for name, (proc, fh, log_path, l_dir, d_dir, t0) in running.items():
        ret = proc.poll()
        if ret is None:
            print(f"   🔄 {name:<39}: {parse_progress(log_path)}")
            backup_to_drive(l_dir, d_dir, synced_set=synced_map[name])
        else:
            fh.close()
            dt = (time.time() - t0) / 60.0
            if ret == 0:
                print(f"   🎉 {name:<39}: XONG 16,000 iters trong {dt:.1f}p! Đang sao lưu lên Drive...")
                backup_to_drive(l_dir, d_dir, synced_set=None)
            else:
                print(f"   ❌ {name:<39}: LỖI (exit={ret})! 30 dòng log cuối:")
                with open(log_path, "r", errors="ignore") as lf:
                    print("".join(lf.readlines()[-30:]))
                failed.append(name)
            finished_names.append(name)
    for n in finished_names:
        running.pop(n)

assert not failed, f"Có model bị lỗi: {failed}"
print(f"\n🏆 HOÀN TẤT 4 MÔ HÌNH BalancedProto-256 (M4, M6, M7, M8) TRONG {(time.time()-t_global)/60.0:.1f} PHÚT!")

### 📊 BƯỚC 3: QUÉT FULL-SCAN `M4, M6, M7, M8` (`BalancedProto-256`, 1,203 ẢNH VAL) & XUẤT BẢNG CHÍNH THỨC ĐỦ 8 DÒNG `M1 → M8`
> Tự động quét 4 bản `BalancedProto-256` mới (~5 phút song song) và **ghép trực tiếp với `M1, M2, M3, M5`** đã hoàn tất trong cùng thư mục Drive để in ra bảng **Full-Factorial $2^3 = 8$ hoàn chỉnh**!

In [ ]:
import os, sys, glob, json, time, subprocess
from prettytable import PrettyTable

%cd /content/TMD/ASCFormer
%env PYTHONPATH=/content/TMD/ASCFormer

DRIVE_ROOT = "/content/drive/MyDrive/RTM_Ablation_B0_16k_bs8_G4"
LOCAL_ROOT = "/content/work_dirs/ablation_b0_16k_bs8_balproto"
BAL_SUITE = [
    ("M4_BalancedProto_256",                  "configs/ascformer/ablation_b0_16k_bs8_m4_balproto.py"),
    ("M6_RealOCR_BalancedProto_256",          "configs/ascformer/ablation_b0_16k_bs8_m6_ocr_balproto.py"),
    ("M7_Focused_Rotate90_BalancedProto_256", "configs/ascformer/ablation_b0_16k_bs8_m7_focused_balproto.py"),
    ("M8_Full_OCR_Focused_BalancedProto_256", "configs/ascformer/ablation_b0_16k_bs8_m8_full_balproto.py"),
]
THRESHOLDS = [
    "0.03", "0.04", "0.05", "0.06", "0.08",
    "0.10", "0.15", "0.20", "0.25", "0.30",
    "0.35", "0.40", "0.45", "0.50", "0.55", "0.60"
]

MAX_PARALLEL_SWEEP = 4
jobs = []
for name, cfg_path in BAL_SUITE:
    d_dir = os.path.join(DRIVE_ROOT, name)
    l_dir = os.path.join(LOCAL_ROOT, name)
    sweep_json = os.path.join(d_dir, f"{name}_sweep.json")
    if os.path.exists(sweep_json):
        print(f"✅ [{name}] Đã có file sweep JSON — bỏ qua!")
        continue
    cands = sorted(glob.glob(f"{l_dir}/iter_16000.pth") + glob.glob(f"{d_dir}/iter_16000.pth"), key=os.path.getmtime)
    if not cands:
        cands = sorted(glob.glob(f"{l_dir}/best_mIoU_*.pth") + glob.glob(f"{d_dir}/best_mIoU_*.pth"), key=os.path.getmtime)
    assert cands, f"❌ Không tìm thấy checkpoint cho {name}!"
    ckpt = cands[-1]
    jobs.append((name, cfg_path, ckpt, sweep_json))

SWEEP_WORKER_PY = "/tmp/run_sweep_worker_balproto.py"
with open(SWEEP_WORKER_PY, "w") as wf:
    wf.write("""import sys, runpy, torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
_orig_load = torch.load
def _safe_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _orig_load(*args, **kwargs)
torch.load = _safe_load
try:
    import yapf.yapflib.yapf_api as _y
    import mmengine.config.config as _m
    _orig_fc = _y.FormatCode
    def _safe_fc(*args, **kwargs):
        kwargs.pop('verify', None)
        return _orig_fc(*args, **kwargs)
    _y.FormatCode = _safe_fc
    _m.FormatCode = _safe_fc
except Exception:
    pass
sys.argv = ['tools/test_threshold_sweep.py'] + sys.argv[1:]
runpy.run_path('tools/test_threshold_sweep.py', run_name='__main__')
""")

running = {}
t0 = time.time()
while jobs or running:
    while jobs and len(running) < MAX_PARALLEL_SWEEP:
        name, cfg_path, ckpt, sweep_json = jobs.pop(0)
        sweep_wd = os.path.join(LOCAL_ROOT, name, "sweep_work")
        cmd = [sys.executable, SWEEP_WORKER_PY, cfg_path, ckpt, "--ann-file", "val.txt", "--work-dir", sweep_wd, "--thresholds", *THRESHOLDS, "--out-json", sweep_json]
        env = os.environ.copy()
        env["PYTHONPATH"] = "/content/TMD/ASCFormer"
        sweep_log_path = os.path.join(os.path.dirname(sweep_json), f"{name}_sweep.log")
        log_f = open(sweep_log_path, "w")
        p = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT, env=env)
        running[name] = (p, log_f, sweep_log_path)
        print(f"🔍 Bắt đầu Full-Scan Sweep 1,203 ảnh: [{name}] ({os.path.basename(ckpt)})")
    time.sleep(15)
    done = []
    for name, (p, log_f, sweep_log_path) in running.items():
        ret = p.poll()
        if ret is not None:
            log_f.close()
            if ret != 0:
                print(f"❌ Sweep lỗi cho {name}! 30 dòng log cuối:")
                with open(sweep_log_path, "r", errors="ignore") as lf:
                    print("".join(lf.readlines()[-30:]))
                raise RuntimeError(f"Sweep failed for {name}")
            print(f"   ✅ Xong Sweep [{name}]!")
            done.append(name)
    for n in done:
        running.pop(n)

print(f"⏱️ Hoàn tất 4 Sweep (BalancedProto) trong {(time.time()-t0)/60.0:.1f} phút!")

# Ghép bảng Full-Factorial 2^3 hoàn chỉnh (M1,M2,M3,M5 + M4,M6,M7,M8 BalancedProto-256)
FULL_8_SPEC = [
    ("M1", "M1_Baseline_2Stream",                    "-", "-", "-"),
    ("M2", "M2_RealOCR_3Stream",                     "✓", "-", "-"),
    ("M3", "M3_Focused_Rotate90",                    "-", "✓", "-"),
    ("M4", "M4_BalancedProto_256",                   "-", "-", "✓"),
    ("M5", "M5_RealOCR_Focused_Rotate90",            "✓", "✓", "-"),
    ("M6", "M6_RealOCR_BalancedProto_256",           "✓", "-", "✓"),
    ("M7", "M7_Focused_Rotate90_BalancedProto_256",  "-", "✓", "✓"),
    ("M8", "M8_Full_OCR_Focused_BalancedProto_256",  "✓", "✓", "✓"),
]

summary = {}
t_std = PrettyTable(["ID", "OCR", "Focus+Rot90", "BalProto-256", "tau", "Tamper IoU (%)", "mIoU (%)", "Prec (%)", "Rec (%)", "F1 (%)"])
t_opt = PrettyTable(["ID", "OCR", "Focus+Rot90", "BalProto-256", "Peak tau*", "Tamper IoU (%)", "mIoU (%)", "Prec (%)", "Rec (%)", "F1 (%)"])

md_std = [
    "# Bảng Tổng Hợp Full-Factorial 2^3 Ablation Study MiT-B0 (BalancedProto-256, 16k × bs=8)\n",
    "- Giao thức: `16,000 iters × bs=8 = 128,000 crops`, `AdamW lr=6e-5`, `PolyLR`, `drop_path=0.1`, `BF16`, pure `CE`, Full-Scan `stride=384` trên 1,203 ảnh `val.txt`.\n",
    "## 1. Ngưỡng chuẩn (tau = 0.500)\n",
    "| ID | OCR | Focus+Rot90 | BalProto-256 | Tamper IoU (%) | mIoU (%) | Precision (%) | Recall (%) | F1 / Dice (%) |",
    "| :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: |",
]
md_opt = [
    "\n## 2. Ngưỡng tối ưu (Peak tau*)\n",
    "| ID | OCR | Focus+Rot90 | BalProto-256 | Peak tau* | Tamper IoU (%) | mIoU (%) | Precision (%) | Recall (%) | F1 / Dice (%) |",
    "| :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: |",
]

for mid, folder, f_ocr, f_foc, f_pro in FULL_8_SPEC:
    sj = os.path.join(DRIVE_ROOT, folder, f"{folder}_sweep.json")
    if not os.path.exists(sj):
        t_std.add_row([mid, f_ocr, f_foc, f_pro, "0.500", "PENDING", "-", "-", "-", "-"])
        t_opt.add_row([mid, f_ocr, f_foc, f_pro, "-", "PENDING", "-", "-", "-", "-"])
        continue
    res = json.load(open(sj))["results"]
    r50 = [r for r in res if abs(r["threshold"] - 0.50) < 1e-4][0]
    rpk = max(res, key=lambda r: r["tampered_iou"])
    summary[folder] = {"id": mid, "factors": {"ocr": f_ocr == "✓", "focused_rot90": f_foc == "✓", "bal_proto": f_pro == "✓"}, "tau_050": r50, "peak": rpk}
    t_std.add_row([mid, f_ocr, f_foc, f_pro, "0.500", f"{r50['tampered_iou']:.2f}", f"{r50['m_iou']:.2f}", f"{r50['tampered_precision']:.2f}", f"{r50['tampered_recall']:.2f}", f"{r50['tampered_dice']:.2f}"])
    t_opt.add_row([mid, f_ocr, f_foc, f_pro, f"{rpk['threshold']:.3f}", f"{rpk['tampered_iou']:.2f}", f"{rpk['m_iou']:.2f}", f"{rpk['tampered_precision']:.2f}", f"{rpk['tampered_recall']:.2f}", f"{rpk['tampered_dice']:.2f}"])
    md_std.append(f"| {mid} | {f_ocr} | {f_foc} | {f_pro} | {r50['tampered_iou']:.2f} | {r50['m_iou']:.2f} | {r50['tampered_precision']:.2f} | {r50['tampered_recall']:.2f} | {r50['tampered_dice']:.2f} |")
    md_opt.append(f"| {mid} | {f_ocr} | {f_foc} | {f_pro} | {rpk['threshold']:.3f} | {rpk['tampered_iou']:.2f} | {rpk['m_iou']:.2f} | {rpk['tampered_precision']:.2f} | {rpk['tampered_recall']:.2f} | {rpk['tampered_dice']:.2f} |")

sum_path = os.path.join(DRIVE_ROOT, "ablation_16k_bs8_FINAL_BALPROTO_8_MODELS_summary.json")
txt_path = os.path.join(DRIVE_ROOT, "ablation_16k_bs8_FINAL_BALPROTO_8_MODELS_report.txt")
md_path  = os.path.join(DRIVE_ROOT, "ablation_16k_bs8_FINAL_BALPROTO_8_MODELS_report.md")

with open(sum_path, "w") as f:
    json.dump(summary, f, indent=2)
with open(txt_path, "w") as f:
    f.write("BẢNG FULL-FACTORIAL 2^3 (M1-M8, BalancedProto-256, 16k x bs=8) TẠI NGƯỠNG CHUẨN (tau = 0.500)\n")
    f.write(str(t_std) + "\n\n")
    f.write("BẢNG FULL-FACTORIAL 2^3 (M1-M8, BalancedProto-256, 16k x bs=8) TẠI NGƯỠNG TỐI ƯU (Peak tau*)\n")
    f.write(str(t_opt) + "\n")
with open(md_path, "w") as f:
    f.write("\n".join(md_std + md_opt) + "\n")

print("\n📌 BẢNG FULL-FACTORIAL 2^3 (BalancedProto-256) TẠI NGƯỠNG CHUẨN (tau = 0.500):")
print(t_std)
print("\n📌 BẢNG FULL-FACTORIAL 2^3 (BalancedProto-256) TẠI NGƯỠNG TỐI ƯU (Peak tau*):")
print(t_opt)
print("💾 Đã lưu báo cáo tổng hợp trên Google Drive:")
print("   - TXT :", txt_path)
print("   - MD  :", md_path)
print("   - JSON:", sum_path)